# 🧠 Mask → Synthetic MRI Demo
Pick a tumor segmentation mask from the dataset, view the **original real FLAIR** image it came from, then generate a **synthetic FLAIR** from the same mask using the trained DDPM.

All outputs are saved to **Google Drive** at `MyDrive/mask-to-mri/demo_outputs/`.

**Requirements:** Colab T4 GPU, Google Drive with checkpoint & dataset.

## 1. Mount Drive & Clone Repo

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_DIR = "/content/drive/MyDrive/mask-to-mri"
REPO_DIR  = "/content/Mask-to-MRI"

# --- Demo output folder on Drive ---
DEMO_OUTPUT_DIR = f"{DRIVE_DIR}/demo_outputs"
os.makedirs(DEMO_OUTPUT_DIR, exist_ok=True)
print(f"Demo outputs will be saved to: {DEMO_OUTPUT_DIR}")

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/AmineAitLaamim/Mask-to-MRI.git
    print("Repository cloned")
else:
    %cd $REPO_DIR
    !git pull
    print("Repository updated")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision tifffile pillow matplotlib numpy tqdm
print("Dependencies installed")

## 3. Extract Dataset to Local Disk

In [ ]:
import shutil

os.makedirs(os.path.join(REPO_DIR, "dataset"), exist_ok=True)

RAW_DIR   = os.path.join(REPO_DIR, "dataset", "lgg-mri-segmentation")
zip_local = "/content/lgg-mri-segmentation.zip"

if os.path.islink(RAW_DIR):
    os.remove(RAW_DIR)

if not os.path.exists(RAW_DIR):
    zip_in_drive = f"{DRIVE_DIR}/dataset/lgg-mri-segmentation.zip"
    if not os.path.exists(zip_in_drive):
        raise FileNotFoundError(f"Dataset zip not found: {zip_in_drive}")

    print(f"Found zip at: {zip_in_drive}")
    if not os.path.exists(zip_local):
        print("Copying zip to local disk...")
        shutil.copy2(zip_in_drive, zip_local)
        print("Copy complete.")

    print("Unzipping to local disk...")
    !unzip -q -o "$zip_local" -d /content/
    print("Unzip complete.")

    extracted_dir = "/content/lgg-mri-segmentation"
    if os.path.exists(extracted_dir):
        if os.path.isdir(RAW_DIR):
            shutil.rmtree(RAW_DIR)
        shutil.move(extracted_dir, RAW_DIR)
        print("Dataset extracted and moved.")
else:
    print(f"Dataset already present: {RAW_DIR}")

print(f"RAW_DIR: {RAW_DIR}")

## 4. Load Best DDPM Model & Dataset Splits

Uses the **best checkpoint: epoch 90 EMA** (val_loss = 0.0232).  
Sampling: **DDIM 250 steps** with **CFG scale 1.5** for sharp tumor boundaries.

In [ ]:
import sys, glob
from pathlib import Path

import numpy as np
import torch
import tifffile
from PIL import Image

# Ensure fresh imports
for _mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[_mod]

sys.path.insert(0, REPO_DIR)

from src.med_ddpm_v3 import ConditionalDDPM, CONFIG
from src.dataset import get_patient_file_list, patient_level_split

CONFIG['raw_dir'] = RAW_DIR
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# --- Load best checkpoint: epoch 90 EMA (val_loss = 0.0232) ---
epoch_90_ckpt = f"{DRIVE_DIR}/outputs_v3/checkpoints/checkpoint_v3_epoch_90.pt"
drive_ckpts   = glob.glob(f"{DRIVE_DIR}/outputs_v3/checkpoints/checkpoint_v3_epoch_*.pt")

if os.path.exists(epoch_90_ckpt):
    ckpt_path = epoch_90_ckpt
    print("Using best checkpoint: epoch 90")
elif drive_ckpts:
    ckpt_path = max(drive_ckpts, key=lambda p: int(Path(p).stem.split('_')[-1]))
    print(f"Epoch 90 not found, falling back to: {os.path.basename(ckpt_path)}")
else:
    raise FileNotFoundError("No v3 checkpoint found in Drive outputs_v3/checkpoints")

model = ConditionalDDPM(CONFIG).to(device)
ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)

if 'ema_state_dict' in ckpt:
    model.load_state_dict(ckpt['ema_state_dict'])
    print(f"Loaded EMA weights from epoch {ckpt['epoch']}")
else:
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded model weights from epoch {ckpt['epoch']}")

model.eval()
print("Model ready ✅")

In [ ]:
# --- Build list of tumor-containing training pairs ---
patient_data = get_patient_file_list(CONFIG['raw_dir'])
splits       = patient_level_split(patient_data, seed=CONFIG['seed'])
train_pairs  = splits['train']

# Keep only slices that actually have a tumor
tumor_pairs = []
for img_path, mask_path in train_pairs:
    m = tifffile.imread(mask_path)
    if np.squeeze(m).max() > 0:
        tumor_pairs.append((img_path, mask_path))

print(f"Training pairs with tumor: {len(tumor_pairs)}")

## 5. Pick a Mask Index
Change `MASK_INDEX` to select a different slice.  
Valid range: **0** to **len(tumor_pairs) - 1**.

In [ ]:
#@title Choose a mask { run: "auto" }
MASK_INDEX = 42  #@param {type:"slider", min:0, max:1100, step:1}

MASK_INDEX = min(MASK_INDEX, len(tumor_pairs) - 1)
img_path, mask_path = tumor_pairs[MASK_INDEX]
stem = Path(mask_path).stem.replace('_mask', '')
print(f"Selected slice: {stem}")
print(f"  Image: {img_path}")
print(f"  Mask:  {mask_path}")

# Create a subfolder for this slice on Drive
slice_output_dir = os.path.join(DEMO_OUTPUT_DIR, stem)
os.makedirs(slice_output_dir, exist_ok=True)
print(f"  Output: {slice_output_dir}")

## 6. Load Original & Generate Synthetic

In [ ]:
# --- Sampling settings (best config from generate_synthetic_improved) ---
DDIM_STEPS = 250
CFG_SCALE  = 1.5

# --- Load the original real image and mask ---
real_img  = tifffile.imread(img_path)        # (H, W, 3) uint8 RGB
real_mask = tifffile.imread(mask_path)        # (H, W) or (H, W, 1)
real_mask = np.squeeze(real_mask)
real_mask_bin = (real_mask > 0).astype(np.uint8) * 255

# Extract FLAIR channel (channel 1 = G) from the real image
real_flair = real_img[:, :, 1]  # single-channel FLAIR

print(f"Real image shape : {real_img.shape}")
print(f"FLAIR channel    : {real_flair.shape}, range [{real_flair.min()}, {real_flair.max()}]")
print(f"Mask shape       : {real_mask_bin.shape}, tumor pixels: {(real_mask_bin > 0).sum()}")

# --- Save mask and real FLAIR to Drive ---
Image.fromarray(real_mask_bin, mode='L').save(os.path.join(slice_output_dir, f"{stem}_mask.png"))
Image.fromarray(real_flair, mode='L').save(os.path.join(slice_output_dir, f"{stem}_real_flair.png"))
print(f"Saved mask and real FLAIR to Drive ✅")

In [ ]:
# --- Prepare mask tensor and generate synthetic FLAIR ---
mask_norm = (real_mask_bin.astype(np.float32) / 127.5) - 1.0    # [-1, 1]
mask_t    = torch.from_numpy(mask_norm).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,H,W)

print(f"Generating synthetic FLAIR (DDIM {DDIM_STEPS} steps, CFG {CFG_SCALE})...")
with torch.no_grad():
    fake = model.sample(mask_t, ddim_steps=DDIM_STEPS, cfg_scale=CFG_SCALE)

# Convert to uint8
synthetic_flair = ((fake[0, 0].cpu().numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
print(f"Synthetic FLAIR  : {synthetic_flair.shape}, range [{synthetic_flair.min()}, {synthetic_flair.max()}]")

# --- Save synthetic FLAIR to Drive ---
Image.fromarray(synthetic_flair, mode='L').save(os.path.join(slice_output_dir, f"{stem}_synthetic_flair.png"))
print(f"Saved synthetic FLAIR to Drive ✅")

## 7. Side-by-Side Comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(real_mask_bin, cmap='gray')
axes[0].set_title('Input Mask', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(real_flair, cmap='gray')
axes[1].set_title('Original Real FLAIR', fontsize=14, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(synthetic_flair, cmap='gray')
axes[2].set_title('Generated Synthetic FLAIR', fontsize=14, fontweight='bold')
axes[2].axis('off')

fig.suptitle(f'Slice: {stem}', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

# Save comparison figure to Drive
fig.savefig(os.path.join(slice_output_dir, f"{stem}_comparison.png"), dpi=150, bbox_inches='tight')
print(f"Saved comparison figure to Drive ✅")
plt.show()

## 8. Overlay: Mask on Real vs Synthetic

In [ ]:
def overlay_mask(image_gray, mask_bin, alpha=0.35):
    """Create RGB overlay with red tumor region."""
    rgb = np.stack([image_gray, image_gray, image_gray], axis=-1).astype(np.float32)
    tumor = mask_bin > 0
    rgb[tumor, 0] = rgb[tumor, 0] * (1 - alpha) + 255 * alpha  # Red channel
    rgb[tumor, 1] = rgb[tumor, 1] * (1 - alpha)                 # Green down
    rgb[tumor, 2] = rgb[tumor, 2] * (1 - alpha)                 # Blue down
    return rgb.clip(0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(overlay_mask(real_flair, real_mask_bin))
axes[0].set_title('Real FLAIR + Mask Overlay', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(overlay_mask(synthetic_flair, real_mask_bin))
axes[1].set_title('Synthetic FLAIR + Mask Overlay', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.suptitle(f'Tumor Overlay — {stem}', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

# Save overlay figure to Drive
fig.savefig(os.path.join(slice_output_dir, f"{stem}_overlay.png"), dpi=150, bbox_inches='tight')
print(f"Saved overlay figure to Drive ✅")
plt.show()

## 9. Generate Multiple Samples from Same Mask
Since DDPM is stochastic, each run produces a different synthetic image for the same mask.  
All individual samples are saved to Drive.

In [ ]:
N_SAMPLES = 4

fig, axes = plt.subplots(1, N_SAMPLES + 2, figsize=(4 * (N_SAMPLES + 2), 4))

# Show mask and real FLAIR first
axes[0].imshow(real_mask_bin, cmap='gray')
axes[0].set_title('Mask', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(real_flair, cmap='gray')
axes[1].set_title('Real FLAIR', fontsize=12, fontweight='bold')
axes[1].axis('off')

# Generate N different synthetic samples and save each
for i in range(N_SAMPLES):
    print(f"  Generating sample {i+1}/{N_SAMPLES}...")
    with torch.no_grad():
        sample_i = model.sample(mask_t, ddim_steps=DDIM_STEPS, cfg_scale=CFG_SCALE)
    sample_np = ((sample_i[0, 0].cpu().numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)

    # Save each individual sample to Drive
    Image.fromarray(sample_np, mode='L').save(
        os.path.join(slice_output_dir, f"{stem}_synthetic_sample_{i+1}.png")
    )

    axes[i + 2].imshow(sample_np, cmap='gray')
    axes[i + 2].set_title(f'Synthetic #{i+1}', fontsize=12, fontweight='bold')
    axes[i + 2].axis('off')

plt.suptitle(f'Multiple Generations — {stem}', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

# Save multi-sample figure to Drive
fig.savefig(os.path.join(slice_output_dir, f"{stem}_multi_samples.png"), dpi=150, bbox_inches='tight')
print(f"\nSaved all samples and multi-sample figure to Drive ✅")
plt.show()

# --- Summary ---
print(f"\n{'='*60}")
print(f"All outputs saved to:")
print(f"  {slice_output_dir}")
print(f"\nFiles:")
for f in sorted(os.listdir(slice_output_dir)):
    print(f"  📄 {f}")
print(f"{'='*60}")